In [1]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_csv("data.csv")

# 날짜 처리 + 정렬
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").drop_duplicates("Date").reset_index(drop=True)

# 퍼센트 문자열 컬럼 안전 변환 (예: '1.2%' -> 0.012)
pct_cols = [c for c in df.columns if "change_%" in c or c.endswith("%")]
for c in pct_cols:
    df[c] = (df[c].astype(str).str.replace("%", "", regex=False).str.replace(",", "", regex=False))
    df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

# 숫자형 안전 변환 (쉼표 제거)
for c in df.columns:
    if c == "Date": 
        continue
    if df[c].dtype == "object":
        df[c] = df[c].astype(str).str.replace(",", "", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")

In [3]:
print("행/열:", df.shape)
display(df.head(3))
display(df.tail(3))

# 타입 요약
display(df.dtypes.to_frame("dtype"))

# 결측 요약
na = df.isna().mean().sort_values(ascending=False)
display(na.to_frame("na_ratio").head(20))


행/열: (2981, 53)


,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield_x,us2y_yield_x,term_spread_x,us10y_yield_y,us2y_yield_y,term_spread_y,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.629997,55.728359,85.919983,27.177940,NaN,NaN,NaN,NaN,3.00,0.39,2.61,3.00,0.39,2.61,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3840,109.356167,0.010935,0.017214,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,NaN,NaN,NaN,NaN,NaN,80.790001,55.523975,85.713120,27.133379,0.001984,-0.003668,-0.002408,-0.001640,3.01,0.41,2.60,3.01,0.41,2.60,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,106.730003,93.430000,13.300003,-0.001497,-0.048583,-0.046031,108.682001,110.1265,109.310667,0.010182,0.012836,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,NaN,NaN,NaN,NaN,NaN,80.650002,55.600620,85.692451,27.031519,-0.001733,0.001380,-0.000241,-0.003754,2.98,0.40,2.58,2.98,0.40,2.58,NaN,NaN,NaN,NaN,NaN


,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield_x,us2y_yield_x,term_spread_x,us10y_yield_y,us2y_yield_y,term_spread_y,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%
2978,2025-11-06,63.380001,59.430000,3.950001,-0.002204,-0.024923,-0.028212,64.260001,63.5395,65.897000,0.005906,0.023667,421168.0,206009.0,206009.0,5202.0,-4729.0,-4729.0,-0.042602,-0.095804,-0.095804,13651.0,13640.0,0.000513,4367.0,5924.0,5604.5,1557.0,86.0,NaN,NaN,NaN,NaN,NaN,99.730003,88.269997,124.250000,41.700001,-0.004691,0.009723,0.007378,0.006031,4.11,3.57,0.54,4.11,3.57,0.54,2063.0,2063.0,2063.0,2063.0,0.0300
2979,2025-11-07,63.630001,59.750000,3.880001,0.003944,-0.022130,0.014347,63.972001,63.5845,65.843500,0.006728,0.017916,421168.0,206009.0,206009.0,5202.0,-4729.0,-4729.0,-0.042602,-0.095804,-0.095804,13651.0,13640.0,0.000513,4367.0,5924.0,5604.5,1557.0,86.0,NaN,NaN,NaN,NaN,NaN,99.599998,89.540001,126.160004,42.439999,-0.001304,0.014388,0.015372,0.017746,NaN,NaN,NaN,NaN,NaN,NaN,2104.0,2104.0,2104.0,2104.0,0.0199
2980,2025-11-10,64.190002,60.349998,3.840004,0.008801,-0.010787,0.013740,63.832001,63.6280,65.815833,0.009042,0.009191,421168.0,206009.0,206009.0,5202.0,-4729.0,-4729.0,-0.042602,-0.095804,-0.095804,13651.0,13640.0,0.000513,4367.0,5924.0,5604.5,1557.0,86.0,NaN,NaN,NaN,NaN,NaN,99.572998,NaN,NaN,NaN,-0.000271,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,dtype
Date,datetime64[ns]
brent_close,float64
wti_close,float64
brent_wti_spread,float64
brent_ret_1d,float64
brent_ret_5d,float64
brent_ret_20d,float64
brent_ma_5,float64
brent_ma_20,float64
brent_ma_60,float64


,na_ratio
wti_mm_net_long,0.795371
wti_sentiment_position,0.795371
wti_mm_net_long_ratio,0.795371
wti_producer_hedge_ratio,0.795371
wti_mm_position_change_wow,0.795371
bdi_open,0.081181
bdi_high,0.081181
bdi_low,0.081181
bdi_change_%,0.081181
bdi_price,0.081181
